## *RaschPy* simulation functionality

This notebook works through examples of how to generate simulated data sets with `RaschPy` for experimental use where knowledge of the underlying 'ground truth' of the generating parameters is useful, for example when comparing the efficacy of different estimation algorithms, such as in Elliott & Buttery (2022a) or exploring the effect of fitting different Rasch models to the same data set, such as in Elliott & Buttery (2022b). There are separate classes for each model: `SLM_Sim` for the simple logistic model (or dichotomous Rasch model) (Rasch, 1960), `PCM_Sim` for the partial credit model (Masters, 1982), `RSM_Sim` for the rating scale model (Andrich, 1978), `MFRM_Sim_Global` for the many-facet Rasch model (Linacre, 1994), and the family of extended MFRMs (Elliott 2025, Elliott & Buttery, 2022b): `MFRM_Sim_Items` for the vector-by-item extended MFRM , `MFRM_Sim_Thresholds` for the vector-by-threshold extended MFRM, `MFRM_Sim_Matrix` for the matrix extended MFRM, and `MFRM_Sim_Bivector` for the bivector extended MFRM. All data is generated to fit the chosen model.

**References**

&nbsp;&nbsp;&nbsp;&nbsp; Andrich, D. (1978). A rating formulation for ordered response categories. *Psychometrika*, *43*(4), 561–573.

&nbsp;&nbsp;&nbsp;&nbsp;   Elliott, M. (2025). Extended many-facet Rasch models: Accounting for rater effects in automated essay scoring systems [Apollo - University of Cambridge Repository]. https://doi.org/10.17863/CAM.127567

&nbsp;&nbsp;&nbsp;&nbsp; Elliott, M., & Buttery, P. J. (2022a) Non-iterative Conditional Pairwise Estimation for the Rating Scale Model, *Educational and Psychological Measurement*, *82*(5), 989-1019.

&nbsp;&nbsp;&nbsp;&nbsp; Elliott, M. and Buttery, P. J. (2022b) Extended Rater Representations in the Many-Facet Rasch Model, *Journal of Applied Measurement*, *22*(1), 133-160.

&nbsp;&nbsp;&nbsp;&nbsp; Linacre, J. M. (1994). *Many-Facet Rasch Measurement*. MESA Press.

&nbsp;&nbsp;&nbsp;&nbsp; Masters, G. N. (1982). A Rasch model for partial credit scoring. *Psychometrika*, *47*(2), 149–174.

&nbsp;&nbsp;&nbsp;&nbsp; Rasch, G. (1960). *Probabilistic models for some intelligence and attainment tests*. Danmarks Pædagogiske
Institut.

Import the packages and set the working directory (here called `my_working_directory`) - you will save your output files here.

In [ ]:
import raschpy as rp
import numpy as np
import pandas as pd
import os

os.chdir('my_working_directory')

### `RSM_Sim`

Create an object `rsm_sim_1` of the class `RSM_Sim` with randomised item difficulties, shared threshold set and person abilities. `RSM_Sim` will do this automatically when you pass `item_range`, `category_base`, `max_disorder`, `person_sd` and `offset` arguments to the simulation: item difficulties will be sampled from a uniform distribution and person abilities will be sampled from a normal distribution. We pass `item_range=4` to have items covering a range of 4 logits, and `person_sd=2` and `offset=1` to have a sample of persons with a mean ability 1 logit higher than the items, with a standard deviation of 2 logits. We also pass the additional arguments `category_base=1.5` and `max_disorder=1`; this sets the base category width to 1.5 logits, with a degree of random uniform variatoin around  controlled by `max_disorder`. With `max_disorder=1`, the minimum category width is 1 logit (and the maximum, symmetrically, will be 2 logits); a smaller value permits more variation in category widths, and a negative value for `max_disorder` allows the presence of disordered thresholds (hence the name of the argument). From this, a set of central item locations are generated from `item_range`, and sets of centred Rasch-ANdrich thresholds, each summing to zero, are generated from  `category_base` and `max_disorder`. One other additional argument that must be passed to `RSM_Sim` is `max_score`, which is a  the maximum possible score for each item. There are 5,000 persons and 12 items, with no missing data for this simulation.

In [ ]:
rsm_sim_1 = rp.RSM_Sim(no_of_items=12,
                       no_of_persons=5000,
                       max_score=5,
                       item_range=4,
                       category_base=1.5,
                       max_disorder=1,
                       person_sd=2,
                       offset=0.5)

Save the generated response dataframe, which is stored as an attribute `rsm_sim_1.responses`, to file, and view the first 5 lines.

In [ ]:
rsm_sim_1.responses.to_csv('rsm_sim_1_responses.csv')
rsm_sim_1.responses.head()

Save the generating item, threshold and person parameters to file, and view the first 5 lines of the item difficulties and the person locations, plus the Rasch-Andrich thresholds.

In [ ]:
rsm_sim_1.items.to_csv('rsm_sim_1_items.csv', header=None)
rsm_sim_1.items.head()

In [ ]:
rsm_sim_1.thresholds.to_csv('rsm_sim_1_thresholds.csv', header=None)
rsm_sim_1.thresholds

In [ ]:
rsm_sim_1.persons.to_csv('rsm_sim_1_persons.csv', header=None)
rsm_sim_1.persons.head()

View `max_score`.

In [ ]:
rsm_sim_1.max_score

Create an object `rsm_1` of the class `RSM` from the response dataframe for analysis. The new object `rsm_1` automatically inherits all the parameters from `rsm_sim_1`, storing them under a namespace `.generating`.

In [ ]:
rsm_1 = rp.RSM(rsm_sim_1)

You may wish to create a simulation based on specified, known item difficulties and/or person locations. This may be done by passing lists to the `manual_items`, `manual_thresholds` and/or `manual_persons` arguments (in which case, there is no need to pass the relevant `item_range`, `category_base`, `max_disorder`, `person_sd` or `offset` arguments). You may also customise the names of the items and/or persons by passing lists of the correct length to the manual_person_names and/or manual_item_names arguments.

The manual_items and manual_persons arguments may also be used to generate random item difficulties and/or person locations according to distributions other than the default uniform (for items) and normal (for persons). This is what is done in the example `rsm_sim_2` below: A set of specified, fixed item difficulties (6 items of difficulty between -2.5 logit and +2.5 logits and a maximum score of 5) and set of Rasch-Andrich thresholds (summing to zero) are passed together with a random uniform distribution of person locations between -2 and 2 logits. For this simulation, we also set a proportion of 20% missing data (missing completely at random) by passing the argument `missing=0.2`.

In [ ]:
rsm_sim_2 = rp.RSM_Sim(no_of_items=6,
                       no_of_persons=1000,
                       max_score=5,
                       missing=0.2,
                       manual_items=[-2.5, -1.5, -0.5, 0.5, 1.5, 2.5],
                       manual_thresholds=[-2, -1, 0, 1, 2],
                       manual_persons = np.random.uniform(-2, 2, 1000))

Save the generated response dataframe, which is stored as an attribute `rsm_sim_2.responses`, to file, and view the first 5 lines.

In [ ]:
rsm_sim_2.responses

Save the generating item, threshold and person parameters to file, and view the item difficulties and Rasch-Andrich thresholds.

In [ ]:
rsm_sim_2.items.to_csv('rsm_sim_2_items.csv', header=None)
rsm_sim_2.items

In [ ]:
rsm_sim_2.thresholds.to_csv('rsm_sim_2_thresholds.csv', header=None)
rsm_sim_2.thresholds

In [ ]:
rsm_sim_2.persons.to_csv('rsm_sim_2_persons.csv', header=None)
rsm_sim_2.persons.head()

View `max_score`.

In [ ]:
rsm_sim_2.max_score

Create an object, `rsm_2`, of the class `RSM` from the response dataframe for analysis.

In [ ]:
rsm_2 = rp.RSM(rsm_sim_2)

The two `RSM` objects `rsm_1` and `rsm_2` are now available for analysis and, where appropriate, comparison of the recovered estmates with the generating estimates. See the example `RSM` notebook for details on how to run an `RSM` analysis.